In [ ]:
!pip install catboost

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


In [ ]:
# Step 1: Load the Base Models (DenseNet121, EfficientNetB0, EfficientNetB3 for Leaf)
leaf_model_1_path = '/content/drive/MyDrive/Capstone Models/DenseNet121_leaf.keras'
leaf_model_2_path = '/content/drive/MyDrive/Capstone Models/EfficientNetB0_leaf.keras'
leaf_model_3_path = '/content/drive/MyDrive/Capstone Models/EfficientNetB3_leaf.keras'

leaf_model_1 = load_model(leaf_model_1_path)
leaf_model_2 = load_model(leaf_model_2_path)
leaf_model_3 = load_model(leaf_model_3_path)

In [ ]:
# Step 2: Load the Dataset (Leaf Test Dataset)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

test_dir = "/content/drive/MyDrive/Capstone Dataset/Guava_Leaf_Dieases/Test"

# Define leaf classes manually
leaf_classes = ['Anthracnose', 'Canker', 'Dot', 'Healthy', 'Rust']  # Replace with actual leaf classes

# Load the test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 1150 files belonging to 5 classes.


In [ ]:
# Step 3: Generate Predictions from Base Models
y_prob_leaf_model_1 = leaf_model_1.predict(test_ds)  # Predictions from DenseNet121
y_prob_leaf_model_2 = leaf_model_2.predict(test_ds)  # Predictions from EfficientNetB0
y_prob_leaf_model_3 = leaf_model_3.predict(test_ds)  # Predictions from EfficientNetB3

36/36 ━━━━━━━━━━━━━━━━━━━━ 212s 5s/step
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 179ms/step
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 245ms/step


In [ ]:
# Step 4: Stack the Predictions (Create Meta-Features)
X_meta_leaf = np.column_stack((
    y_prob_leaf_model_1,
    y_prob_leaf_model_2,
    y_prob_leaf_model_3
))

# True labels for the test set (adjust this to match your dataset)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

# Step 5: Define and Train Meta-Models (Each Section)

In [ ]:
## 1. Logistic Regression
print("\nTraining Logistic Regression...")
meta_model_lr = LogisticRegression()
meta_model_lr.fit(X_meta_leaf, y_true)
y_meta_pred_lr = meta_model_lr.predict(X_meta_leaf)
print(f"Logistic Regression Accuracy: {accuracy_score(y_true, y_meta_pred_lr):.4f}")
print(classification_report(y_true, y_meta_pred_lr, target_names=leaf_classes))


Training Logistic Regression...
Logistic Regression Accuracy: 0.8217
              precision    recall  f1-score   support

 Anthracnose       0.80      0.92      0.85       230
      Canker       0.91      0.83      0.87       230
         Dot       0.67      0.62      0.64       230
     Healthy       0.82      0.90      0.86       230
        Rust       0.92      0.83      0.87       230

    accuracy                           0.82      1150
   macro avg       0.82      0.82      0.82      1150
weighted avg       0.82      0.82      0.82      1150



In [ ]:
## 2. Random Forest
print("\nTraining Random Forest...")
meta_model_rf = RandomForestClassifier()
meta_model_rf.fit(X_meta_leaf, y_true)
y_meta_pred_rf = meta_model_rf.predict(X_meta_leaf)
print(f"Random Forest Accuracy: {accuracy_score(y_true, y_meta_pred_rf):.4f}")
print(classification_report(y_true, y_meta_pred_rf, target_names=leaf_classes))


Training Random Forest...
Random Forest Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 3. Gradient Boosting
print("\nTraining Gradient Boosting...")
meta_model_gb = GradientBoostingClassifier()
meta_model_gb.fit(X_meta_leaf, y_true)
y_meta_pred_gb = meta_model_gb.predict(X_meta_leaf)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_true, y_meta_pred_gb):.4f}")
print(classification_report(y_true, y_meta_pred_gb, target_names=leaf_classes))


Training Gradient Boosting...
Gradient Boosting Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 4. Support Vector Classifier (SVC)
print("\nTraining Support Vector Classifier (SVC)...")
meta_model_svc = SVC()
meta_model_svc.fit(X_meta_leaf, y_true)
y_meta_pred_svc = meta_model_svc.predict(X_meta_leaf)
print(f"SVC Accuracy: {accuracy_score(y_true, y_meta_pred_svc):.4f}")
print(classification_report(y_true, y_meta_pred_svc, target_names=leaf_classes))


Training Support Vector Classifier (SVC)...
SVC Accuracy: 0.8774
              precision    recall  f1-score   support

 Anthracnose       0.84      0.93      0.88       230
      Canker       0.92      0.94      0.93       230
         Dot       0.83      0.70      0.76       230
     Healthy       0.85      0.95      0.90       230
        Rust       0.96      0.87      0.91       230

    accuracy                           0.88      1150
   macro avg       0.88      0.88      0.88      1150
weighted avg       0.88      0.88      0.88      1150



In [ ]:
## 5. Multi-layer Perceptron (MLP)
print("\nTraining Multi-layer Perceptron (MLP)...")
meta_model_mlp = MLPClassifier()
meta_model_mlp.fit(X_meta_leaf, y_true)
y_meta_pred_mlp = meta_model_mlp.predict(X_meta_leaf)
print(f"MLP Accuracy: {accuracy_score(y_true, y_meta_pred_mlp):.4f}")
print(classification_report(y_true, y_meta_pred_mlp, target_names=leaf_classes))


Training Multi-layer Perceptron (MLP)...
MLP Accuracy: 0.8826
              precision    recall  f1-score   support

 Anthracnose       0.86      0.96      0.91       230
      Canker       0.92      0.94      0.93       230
         Dot       0.82      0.69      0.75       230
     Healthy       0.85      0.92      0.88       230
        Rust       0.96      0.90      0.93       230

    accuracy                           0.88      1150
   macro avg       0.88      0.88      0.88      1150
weighted avg       0.88      0.88      0.88      1150



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
## 6. AdaBoost Classifier
print("\nTraining AdaBoost Classifier...")
meta_model_ada = AdaBoostClassifier()
meta_model_ada.fit(X_meta_leaf, y_true)
y_meta_pred_ada = meta_model_ada.predict(X_meta_leaf)
print(f"AdaBoost Accuracy: {accuracy_score(y_true, y_meta_pred_ada):.4f}")
print(classification_report(y_true, y_meta_pred_ada, target_names=leaf_classes))


Training AdaBoost Classifier...
AdaBoost Accuracy: 0.8322
              precision    recall  f1-score   support

 Anthracnose       0.76      0.93      0.83       230
      Canker       0.88      0.90      0.89       230
         Dot       0.74      0.57      0.64       230
     Healthy       0.85      0.93      0.89       230
        Rust       0.95      0.83      0.89       230

    accuracy                           0.83      1150
   macro avg       0.83      0.83      0.83      1150
weighted avg       0.83      0.83      0.83      1150



In [ ]:
## 7. Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
meta_model_dt = DecisionTreeClassifier()
meta_model_dt.fit(X_meta_leaf, y_true)
y_meta_pred_dt = meta_model_dt.predict(X_meta_leaf)
print(f"Decision Tree Accuracy: {accuracy_score(y_true, y_meta_pred_dt):.4f}")
print(classification_report(y_true, y_meta_pred_dt, target_names=leaf_classes))


Training Decision Tree Classifier...
Decision Tree Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 8. K-Nearest Neighbors (KNN)
print("\nTraining K-Nearest Neighbors (KNN)...")
meta_model_knn = KNeighborsClassifier()
meta_model_knn.fit(X_meta_leaf, y_true)
y_meta_pred_knn = meta_model_knn.predict(X_meta_leaf)
print(f"KNN Accuracy: {accuracy_score(y_true, y_meta_pred_knn):.4f}")
print(classification_report(y_true, y_meta_pred_knn, target_names=leaf_classes))


Training K-Nearest Neighbors (KNN)...
KNN Accuracy: 0.9426
              precision    recall  f1-score   support

 Anthracnose       0.94      0.96      0.95       230
      Canker       0.98      0.95      0.96       230
         Dot       0.88      0.90      0.89       230
     Healthy       0.94      0.94      0.94       230
        Rust       0.97      0.96      0.96       230

    accuracy                           0.94      1150
   macro avg       0.94      0.94      0.94      1150
weighted avg       0.94      0.94      0.94      1150



In [ ]:
## 9. Gaussian Naive Bayes (GNB)
print("\nTraining Gaussian Naive Bayes (GNB)...")
meta_model_gnb = GaussianNB()
meta_model_gnb.fit(X_meta_leaf, y_true)
y_meta_pred_gnb = meta_model_gnb.predict(X_meta_leaf)
print(f"GNB Accuracy: {accuracy_score(y_true, y_meta_pred_gnb):.4f}")
print(classification_report(y_true, y_meta_pred_gnb, target_names=leaf_classes))


Training Gaussian Naive Bayes (GNB)...
GNB Accuracy: 0.7983
              precision    recall  f1-score   support

 Anthracnose       0.78      0.90      0.83       230
      Canker       0.89      0.82      0.85       230
         Dot       0.58      0.57      0.57       230
     Healthy       0.85      0.91      0.88       230
        Rust       0.91      0.80      0.85       230

    accuracy                           0.80      1150
   macro avg       0.80      0.80      0.80      1150
weighted avg       0.80      0.80      0.80      1150



In [ ]:
## 10. Quadratic Discriminant Analysis (QDA)
print("\nTraining Quadratic Discriminant Analysis (QDA)...")
meta_model_qda = QuadraticDiscriminantAnalysis()
meta_model_qda.fit(X_meta_leaf, y_true)
y_meta_pred_qda = meta_model_qda.predict(X_meta_leaf)
print(f"QDA Accuracy: {accuracy_score(y_true, y_meta_pred_qda):.4f}")
print(classification_report(y_true, y_meta_pred_qda, target_names=leaf_classes))


Training Quadratic Discriminant Analysis (QDA)...
QDA Accuracy: 0.8583
              precision    recall  f1-score   support

 Anthracnose       0.84      0.92      0.88       230
      Canker       0.92      0.90      0.91       230
         Dot       0.74      0.66      0.70       230
     Healthy       0.86      0.93      0.90       230
        Rust       0.92      0.88      0.90       230

    accuracy                           0.86      1150
   macro avg       0.86      0.86      0.86      1150
weighted avg       0.86      0.86      0.86      1150



/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/p

In [ ]:
## 11. XGBoost Classifier
print("\nTraining XGBoost Classifier...")
meta_model_xgb = XGBClassifier()
meta_model_xgb.fit(X_meta_leaf, y_true)
y_meta_pred_xgb = meta_model_xgb.predict(X_meta_leaf)
print(f"XGBoost Accuracy: {accuracy_score(y_true, y_meta_pred_xgb):.4f}")
print(classification_report(y_true, y_meta_pred_xgb, target_names=leaf_classes))


Training XGBoost Classifier...
XGBoost Accuracy: 1.0000
              precision    recall  f1-score   support

 Anthracnose       1.00      1.00      1.00       230
      Canker       1.00      1.00      1.00       230
         Dot       1.00      1.00      1.00       230
     Healthy       1.00      1.00      1.00       230
        Rust       1.00      1.00      1.00       230

    accuracy                           1.00      1150
   macro avg       1.00      1.00      1.00      1150
weighted avg       1.00      1.00      1.00      1150



In [ ]:
## 12. CatBoost Classifier
print("\nTraining CatBoost Classifier...")
meta_model_cat = CatBoostClassifier(learning_rate=0.1, iterations=1000, depth=6)
meta_model_cat.fit(X_meta_leaf, y_true)
y_meta_pred_cat = meta_model_cat.predict(X_meta_leaf)
print(f"CatBoost Accuracy: {accuracy_score(y_true, y_meta_pred_cat):.4f}")
print(classification_report(y_true, y_meta_pred_cat, target_names=leaf_classes))


Training CatBoost Classifier...
0:	learn: 1.4378847	total: 64.5ms	remaining: 1m 4s
1:	learn: 1.3095187	total: 77.5ms	remaining: 38.7s
2:	learn: 1.1884225	total: 89.9ms	remaining: 29.9s
3:	learn: 1.0981423	total: 102ms	remaining: 25.4s
4:	learn: 1.0172123	total: 114ms	remaining: 22.7s
5:	learn: 0.9525118	total: 126ms	remaining: 20.9s
6:	learn: 0.8962121	total: 138ms	remaining: 19.6s
7:	learn: 0.8442659	total: 151ms	remaining: 18.7s
8:	learn: 0.7996563	total: 164ms	remaining: 18s
9:	learn: 0.7581724	total: 179ms	remaining: 17.7s
10:	learn: 0.7244509	total: 192ms	remaining: 17.3s
11:	learn: 0.6929704	total: 204ms	remaining: 16.8s
12:	learn: 0.6649572	total: 216ms	remaining: 16.4s
13:	learn: 0.6385235	total: 229ms	remaining: 16.1s
14:	learn: 0.6160976	total: 242ms	remaining: 15.9s
15:	learn: 0.5959819	total: 254ms	remaining: 15.7s
16:	learn: 0.5775595	total: 267ms	remaining: 15.4s
17:	learn: 0.5566774	total: 279ms	remaining: 15.2s
18:	learn: 0.5412274	total: 292ms	remaining: 15.1s
19:	lea

In [ ]:
## 13. LightGBM Classifier
print("\nTraining LightGBM Classifier...")
meta_model_lgbm = LGBMClassifier()
meta_model_lgbm.fit(X_meta_leaf, y_true)
y_meta_pred_lgbm = meta_model_lgbm.predict(X_meta_leaf)
print(f"LightGBM Accuracy: {accuracy_score(y_true, y_meta_pred_lgbm):.4f}")
print(classification_report(y_true, y_meta_pred_lgbm, target_names=leaf_classes))


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000292 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 1150, number of used features: 15
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
